In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("spotify_millsongdata.csv")

In [3]:
df.head(5)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [4]:
df.tail(5)

,artist,song,link,text
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...
57649,Zwan,Heartsong,/z/zwan/heartsong_20148991.html,come in \r\nmake yourself at home \r\ni'm a ...


In [5]:
df.shape

(57650, 4)

In [6]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [7]:
df=df.sample(5000).drop('link',axis=1).reset_index(drop=True)

In [8]:
df.head(10)

,artist,song,text
0,John Martyn,Sapphire,I watch the day go down \r\nSapphire \r\nI w...
1,Unwritten Law,Welcome To Oblivion,You're the chemical I'm in trouble I've lost a...
2,Justin Bieber,Die In Your Arms,"Mhm, uh huh, yeah yeah, alright, \r\nMhm, uh ..."
3,Weird Al Yankovic,Midnight Star,I was waiting in the express lane with my twel...
4,Beautiful South,My Book,This is my life and this is how it reads \r\n...
5,Oscar Hammerstein,All Er Nothin',You'll have to be a little more standoffish \...
6,Donna Summer,The Christmas Song,Chestnuts roasting on an open fire \r\nJack f...
7,Slayer,Darkness Of Christ,Mankind in his insatiable search for divine \...
8,Patti Smith,Dead City,This dead city \r\nLongs to be \r\nThis dead...
9,Judy Garland,Boys And Girls Like You And Me,We walk on every city street \r\nWe walk in l...


In [9]:
df['text'][0]

"I watch the day go down  \r\nSapphire  \r\nI watch my luck turn round  \r\nA high flyer  \r\nI threw my bones around  \r\nSure fire  \r\nI watch the current run  \r\nLive wire  \r\nClear blue  \r\nToo true  \r\nClear blue  \r\nToo true.  \r\n  \r\nI don't know what to do  \r\nI got no place to go  \r\nOh the day I lost my sweet Sapphire  \r\nMy precious gems are dust  \r\nThere's nothing left to trust  \r\nOh the days I'll miss sweet Sapphire.  \r\n  \r\nI saw her running round  \r\nSweet liar.  \r\nAnd I ran the garden path  \r\nSweet briar  \r\nI did just what I could  \r\nSo tired  \r\nI threw my keys away  \r\nWith no desire  \r\nClear blue  \r\nToo true.  \r\n  \r\nI don't know what to do  \r\nI got no place to go  \r\nOh the days I miss sweet Sapphire  \r\nMy precious gems are dust  \r\nThere's nothing left to trust  \r\nOh the days I'll miss sweet Sapphire.  \r\n  \r\nI got no place to go  \r\nI don't know what to do  \r\nOoh, the dream about Sapphire  \r\nI don't know what to 

In [12]:
df.shape

(5000, 3)

In [13]:
df['text']=df['text'].str.lower().replace(r'^\w\s',' ').replace(r'\n',' ', regex=True)

In [14]:
import nltk
from nltk.stem.porter import PorterStemmer
stemmer=PorterStemmer()

def tokenization(txt):
    tokens=nltk.word_tokenize(txt)
    stemming=[stemmer.stem(w) for w in tokens]
    return " ".join(stemming)

In [15]:
df['text']=df['text'].apply(lambda x: tokenization(x))

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
tfidvector= TfidfVectorizer(analyzer='word', stop_words='english')
matrix=tfidvector.fit_transform(df['text'])
similarity=cosine_similarity(matrix)

In [18]:
similarity[0]

array([1.        , 0.03144783, 0.02681517, ..., 0.00980952, 0.0218162 ,
       0.00859978], shape=(5000,))

In [19]:
df[df['song']=='Crying Over You']

,artist,song,text
4183,ABBA,Crying Over You,i 'm waitin ' for you babi i 'm sit all alon i...


In [20]:
def recommendation(song_df):
    idx=df[df['song']==song_df].index[0]
    distances=sorted(list(enumerate(similarity[idx])), reverse=True,key=lambda x:x[1])

    songs=[]
    for m_id in distances[1:21]:
        songs.append(df.iloc[m_id[0]].song)
    return songs

In [21]:
recommendation('Crying Over You')

["Cryin' Time",
 'My Lovely Man',
 'Cryin Shame',
 'Hold Back The Tears',
 'I Want You To Want Me',
 'Please Send Me Someone To Love',
 'People At The Top',
 'Oh Draw Me Lord',
 'Blue, Blue Day',
 'Color Of The Blues',
 'Good Morning Blues',
 'Blue Jean Blues',
 'Strength Of My Life',
 'The Appeal',
 'Blue Christmas',
 'Cool Blue',
 'In The Blue',
 'Consuming Fire',
 'Am I Blue',
 'Black And Blue']

In [22]:
import pickle 
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(df,open('df.pkl','wb'))